# Notebook 02 — Implementación de programación dinámica

Acompaña la sección 21.4. Vas a implementar las tres versiones (ingenua, memoizada, tabulada) del cálculo de la función de valor, **medir** cuántas operaciones hace cada una, y *ver* por qué DP es categóricamente distinta de la versión directa.

## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time


def make_costs(n, seed=42):
    """Genera un vector de costos para un problema de tamaño n."""
    rng = np.random.RandomState(seed)
    c = rng.randint(1, 10, size=n).astype(float)
    c[-1] = 0  # la meta no cuesta
    return c


# Escalera base (misma que el módulo)
COSTS_BASE = np.array([3, 2, 5, 10, 1, 0], dtype=float)


## Versión 1 — Recursión ingenua

La implementación directa de la ecuación de Bellman. Sin caché, sin memoria: cada llamada recalcula todo desde cero. Vamos a contar cuántas veces se llama a sí misma.


In [ ]:
call_count_naive = 0


def optimal_naive(i, costs):
    global call_count_naive
    call_count_naive += 1
    n = len(costs)
    if i >= n - 1:
        return 0.0
    best = costs[i] + optimal_naive(i + 1, costs)
    if i + 2 < n:
        best = min(best, costs[i] + optimal_naive(i + 2, costs))
    return best


# Correr para varios N y medir llamadas
print("Recursión ingenua — número de llamadas:")
print(f"  {'N':>4s}  {'Llamadas':>12s}")
for N in [5, 10, 15, 20, 25]:
    costs = make_costs(N)
    call_count_naive = 0
    result = optimal_naive(0, costs)
    print(f"  {N:>4d}  {call_count_naive:>12,d}")


Los números crecen **exponencialmente**. Para $N = 25$ ya estamos haciendo miles de llamadas para un problema que en el fondo tiene solo 25 subproblemas distintos.


## Versión 2 — Memoización (top-down con caché)

Misma función recursiva, pero con un caché.


In [ ]:
from functools import lru_cache


def optimal_memo(costs):
    n = len(costs)
    count = [0]

    @lru_cache(maxsize=None)
    def rec(i):
        count[0] += 1   # cache misses only — los aciertos del caché no pasan por el cuerpo
        if i >= n - 1:
            return 0.0
        best = costs[i] + rec(i + 1)
        if i + 2 < n:
            best = min(best, costs[i] + rec(i + 2))
        return best

    value = rec(0)
    return value, count[0]


# Correr y medir
print("Memoización — número de llamadas (cada subproblema se calcula una vez):")
print(f"  {'N':>4s}  {'Llamadas':>12s}")
for N in [5, 10, 15, 20, 25, 30]:
    costs = make_costs(N)
    _, calls = optimal_memo(costs)
    print(f"  {N:>4d}  {calls:>12,d}")


La memoización cachea los resultados. Cada subproblema se resuelve una sola vez. El conteo ahora crece **linealmente** con $N$.


## Versión 3 — Tabulación (bottom-up)

Sin recursión. Un loop que llena el arreglo de derecha a izquierda.


In [ ]:
def optimal_tab(costs):
    n = len(costs)
    V = np.zeros(n)
    pi = [None] * n
    for i in range(n - 2, -1, -1):
        # Opción 1: subir 1
        options = [("subir 1", costs[i] + V[i + 1])]
        # Opción 2: saltar 2 (si aplica)
        if i + 2 < n:
            options.append(("saltar 2", costs[i] + V[i + 2]))
        action, value = min(options, key=lambda t: t[1])
        V[i] = value
        pi[i] = action
    return V, pi


V_tab, pi_tab = optimal_tab(COSTS_BASE)
print("V =", V_tab.tolist())
print("π =", pi_tab)
print(f"V(0) = {V_tab[0]}")


## Verificar que las tres dan el mismo resultado


In [ ]:
# Limpieza del contador global
call_count_naive = 0

v_naive = optimal_naive(0, COSTS_BASE)
v_memo, _ = optimal_memo(COSTS_BASE)
v_tab_0 = optimal_tab(COSTS_BASE)[0][0]

print(f"V(0) ingenua:    {v_naive}")
print(f"V(0) memoizada:  {v_memo}")
print(f"V(0) tabulada:   {v_tab_0}")

assert v_naive == v_memo == v_tab_0 == 9
print("\n✓ Las tres implementaciones dan V(0) = 9. Misma ecuación, distintos recorridos.")


## Medir la brecha exponencial vs. lineal

Vamos a contar operaciones para varios valores de $N$ y graficar.


In [ ]:
# Conteo para la versión ingenua (sin caché)
def count_calls_naive(n):
    costs = make_costs(n)
    counter = [0]
    def rec(i):
        counter[0] += 1
        if i >= n - 1:
            return 0.0
        best = costs[i] + rec(i + 1)
        if i + 2 < n:
            best = min(best, costs[i] + rec(i + 2))
        return best
    rec(0)
    return counter[0]


def count_calls_dp(n):
    """Tabulación: un cálculo por estado."""
    return n


N_range = list(range(2, 28))
naive_counts = [count_calls_naive(n) for n in N_range]
dp_counts = [count_calls_dp(n) for n in N_range]

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(N_range, naive_counts, marker="o", color="#E94F37",
            linewidth=2, markersize=5, label="Recursión ingenua")
ax.semilogy(N_range, dp_counts, marker="s", color="#27AE60",
            linewidth=2, markersize=5, label="DP (tabulación / memoización)")
ax.set_xlabel("tamaño del problema N")
ax.set_ylabel("número de operaciones (escala log)")
ax.set_title("Exponencial vs. lineal: la diferencia es categórica")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPara N = {N_range[-1]}:")
print(f"  Ingenua: {naive_counts[-1]:,} llamadas")
print(f"  DP:      {dp_counts[-1]:,} operaciones")
print(f"  Factor:  ~{naive_counts[-1] / dp_counts[-1]:,.0f}×")


## Medir tiempo real de ejecución


In [ ]:
def time_naive(n, repeats=1):
    costs = make_costs(n)
    t0 = time.perf_counter()
    for _ in range(repeats):
        optimal_naive(0, costs)
    return (time.perf_counter() - t0) / repeats


def time_dp(n, repeats=100):
    costs = make_costs(n)
    t0 = time.perf_counter()
    for _ in range(repeats):
        optimal_tab(costs)
    return (time.perf_counter() - t0) / repeats


print(f"{'N':>4s}  {'Ingenua (s)':>15s}  {'DP (s)':>15s}")
for N in [10, 20, 25]:
    t_naive = time_naive(N)
    t_dp = time_dp(N)
    print(f"{N:>4d}  {t_naive:>15.6f}  {t_dp:>15.8f}")


## Resumen

- La **misma ecuación de Bellman** se puede recorrer de tres formas: ingenua, memoizada (top-down), tabulada (bottom-up).
- Las tres dan el mismo $V(0)$. La diferencia es cuántas veces calculan cada subproblema.
- La brecha entre ingenua y DP es exponencial en el tamaño del problema.
- Para problemas grandes ($N \gtrsim 30$), la versión ingenua es inviable. La DP termina en milisegundos.

Esto es, literalmente, la diferencia entre tener una **ecuación** (la de Bellman) y tener un **algoritmo** (programación dinámica). La ecuación sin el algoritmo no se resuelve; el algoritmo sin la ecuación no existe.
